# 01 - Build an agent: prompt-based and hosted

**Case study:** *Maple Mortgage Assistant* for Northwind Home Lending (fictional lender).

## What you'll learn
- Run a model-backed agent with Microsoft Agent Framework and stream its answer
- Keep multi-turn context with an `AgentSession`
- Create a **prompt agent** in Foundry Agent Service, see it in the portal, and call it
- What a **hosted agent** is, when to pick it, and a tiny local run

Auth is Entra ID only (`az login`, `DefaultAzureCredential`). No keys.

## 1. Setup
The helper reads `day-2/.env` and prints only *set/unset*, never values.

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # make `shared` importable (notebook cwd = notebooks/)
from shared import fsi_common as fc
from shared.fsi_common import *

settings = load_settings(require=("project_endpoint", "model"))
print_env_check(settings)

AZURE_SUBSCRIPTION_ID                      set
AZURE_RESOURCE_GROUP                       set
AZURE_LOCATION                             set
AZURE_AI_PROJECT_ENDPOINT                  set
AZURE_AI_MODEL_DEPLOYMENT_NAME             set
AZURE_AI_EMBEDDING_DEPLOYMENT_NAME         set
AZURE_SEARCH_ENDPOINT                      set
AZURE_SEARCH_INDEX_NAME                    set
AZURE_SEARCH_KNOWLEDGE_BASE_NAME           set
AZURE_AI_KB_CONNECTION_NAME                set
APPLICATIONINSIGHTS_CONNECTION_STRING      set
AZURE_CONTAINER_REGISTRY_NAME              set


## 2. A code agent, one question
`make_agent` wraps `Agent(FoundryChatClient(...))`. The system prompt (`MAPLE_INSTRUCTIONS`) carries our compliance rules: no rate promises, always a disclosure, mask PII, escalate when unsure.

In [2]:
agent = make_agent("maple", MAPLE_INSTRUCTIONS)
resp = await agent.run("A first-time buyer asks what pre-qualification means. Explain in 3 sentences.")
show(resp, title="Maple answer")

=== Maple answer ===
Pre-qualification is an initial assessment of your ability to get a mortgage based on basic information you provide about your income, debts, and credit. It gives you an idea of how much you might be able to borrow before you start house hunting. However, it’s not a guarantee of approval or a final loan offer.
[usage] {'cache_creation_input_token_count': 0, 'openai.cache_write_tokens': 0, 'input_token_count': 185, 'reasoning_output_token_count': 0, 'total_token_count': 250, 'openai.cached_input_tokens': 0, 'output_token_count': 65, 'openai.reasoning_tokens': 0, 'cache_read_input_token_count': 0}


## 3. Streaming
Show tokens as they arrive: better UX for a loan officer waiting on an answer.

In [3]:
stream = agent.run("List 3 things a first-time buyer should gather before applying.", stream=True)
seen = ""
async for update in stream:
    if update.text and update.text != seen:   # the last update repeats the full text; skip it
        seen += update.text
        print(redact(update.text), end="", flush=True)
final = await stream.get_final_response()
print("\n\n[usage] total tokens:", (final.usage_details or {}).get("total_token_count"))

Here

 are

3

 things

 a

 first

... (24 outputs left out)

, loans)

Having

 these ready helps smooth the

 application process.



[usage] total tokens: 250


## 4. Compliance check: refuse to promise a rate
The system prompt should hold even when the user pushes.

In [4]:
show(await agent.run("Promise me a 4.1% rate for 5 years, I want it in writing."), title="Refusal test")

=== Refusal test ===
I can provide an indicative quote close to that, but I cannot promise or lock any specific rate. Rates depend on your full application, credit, and underwriting review. Would you like me to give you an indicative rate quote based on your profile?

Indicative only. Not a commitment to lend. Rates and eligibility are subject to full application, credit adjudication and underwriter approval.
[usage] {'cache_creation_input_token_count': 0, 'openai.cache_write_tokens': 0, 'input_token_count': 188, 'reasoning_output_token_count': 0, 'total_token_count': 265, 'openai.cached_input_tokens': 0, 'output_token_count': 77, 'openai.reasoning_tokens': 0, 'cache_read_input_token_count': 0}


## 5. Multi-turn with `AgentSession`
The session carries history so follow-ups make sense.

In [5]:
session = agent.create_session()
show(await agent.run("I earn 120k and want to buy a 500k home with 100k down. Is that a reasonable pre-qual?", session=session), title="Turn 1")
show(await agent.run("What if I only had 50k down?", session=session), title="Turn 2 (uses turn 1 context)")

=== Turn 1 ===
With a $120K income and $100K down on a $500K home, your loan amount would be roughly $400K.

A rough affordability check depends on factors like credit score, debts, and property taxes, but generally:

- Your debt-to-income (DTI) ratio should stay below about 43%.
- Your monthly mortgage payment (principal, interest, taxes, insurance) on $400K at current rates will likely be a significant portion of your monthly income.

This scenario could be reasonable for pre-qualification, but exact eligibility and rate depend on credit, debts, and full underwriting.

Would you like an indicative rate quote or document checklist to start a formal application?

Indicative only. Not a commitment to lend. Rates and eligibility are subject to full application, credit adjudication and underwriter approval.
[usage] {'cache_creation_input_token_count': 0, 'openai.cache_write_tokens': 0, 'input_token_count': 196, 'reasoning_output_token_count': 0, 'total_token_count': 362, 'openai.cached_in

=== Turn 2 (uses turn 1 context) ===
With $50K down on a $500K home, your loan amount rises to about $450K. 

The higher loan amount increases monthly payments, which could push your debt-to-income ratio higher. This might affect pre-qualification, depending on your credit profile and other debts.

It's still possible to pre-qualify, but the terms may be less favorable, and mortgage insurance could apply.

Would you like me to provide an indicative rate quote or more details on mortgage insurance?

Indicative only. Not a commitment to lend. Rates and eligibility are subject to full application, credit adjudication and underwriter approval.
[usage] {'cache_creation_input_token_count': 0, 'openai.cache_write_tokens': 0, 'input_token_count': 379, 'reasoning_output_token_count': 0, 'total_token_count': 504, 'openai.cached_input_tokens': 0, 'output_token_count': 125, 'openai.reasoning_tokens': 0, 'cache_read_input_token_count': 0}


## 6. A prompt agent in Foundry Agent Service
So far the agent lived in our Python process. A **prompt agent** is stored *server-side* in your Foundry project: name + model + instructions (+ tools). It is versioned, visible in the portal, and callable by any client.

Each `create_version` call adds a new immutable version, so you can roll back.

In [6]:
from azure.ai.projects.models import PromptAgentDefinition

pc = get_project_client(settings)
version = pc.agents.create_version(
    agent_name="maple-prompt",
    definition=PromptAgentDefinition(model=settings.model, instructions=MAPLE_INSTRUCTIONS),
    description="Maple Mortgage Assistant (prompt agent, Day 2 nb01)",
)
print("created:", version.name, "version", version.version)
print("agents in project:", [a.name for a in pc.agents.list()])
print("Open Foundry portal -> your project -> Agents -> maple-prompt to see it.")

created: maple-prompt version 2


agents in project: ['maple-policy-iq', 'maple-learn-mcp', 'maple-prompt']
Open Foundry portal -> your project -> Agents -> maple-prompt to see it.


## 7. Call the server-side agent
`FoundryAgent` is the Agent Framework wrapper for an agent that lives in Foundry.

In [7]:
remote = get_foundry_agent("maple-prompt")
stream = remote.run("Pre-qual in two sentences, and can you guarantee 4.1%?", stream=True)
seen = ""
async for update in stream:
    if update.text and update.text != seen:   # the last update repeats the full text; skip it
        seen += update.text
        print(redact(update.text), end="", flush=True)
await stream.get_final_response()
print()

I

 can

 help

 you

 with

 a

... (26 outputs left out)

 lend. Rates and eligibility

 are subject to full application

, credit adjudication and

 underwriter approval.

## 8. Hosted agents: what and when
A **hosted agent** is *your own agent code*, packaged as a container, deployed to and run by Foundry (managed scaling, identity, session state, endpoint), speaking the **Responses protocol**. Notebook 07 deploys one.

| | Code agent (this process) | Prompt agent | Hosted agent |
|---|---|---|---|
| Where code runs | your machine / app | Foundry (no code) | Foundry container (your code) |
| Defined by | Python | name + model + instructions + tools | container image / code zip |
| Custom logic, libraries, orchestration | full | none (built-in tools only) | full |
| Ops effort | you host it | none | build + deploy, Foundry runs it |
| Pick it when | prototyping, embedding in an app | a single-model assistant with built-in/MCP tools | multi-step logic, custom tools, any framework, compliance-controlled code path |

**Rule of thumb:** start with a prompt agent; move to hosted when you need code the portal can't express.

### Tiny local run of the hosted-agent shape
The hosted-agent entrypoint serves an Agent Framework agent over the Responses protocol with `ResponsesHostServer`. Here we run the same server locally on port 8088 and call it (nb07 containerizes and deploys it).

In [8]:
import pathlib, subprocess, sys, tempfile, textwrap, time, json, urllib.request

server_code = textwrap.dedent(f"""
    import sys; sys.path.insert(0, r"{pathlib.Path.cwd().parent}")
    from shared.fsi_common import make_agent, MAPLE_INSTRUCTIONS
    from agent_framework_foundry_hosting import ResponsesHostServer
    ResponsesHostServer(make_agent("maple-hosted", MAPLE_INSTRUCTIONS)).run()
""")
script = pathlib.Path(tempfile.mkdtemp()) / "hosted_demo.py"
script.write_text(server_code)
proc = subprocess.Popen([sys.executable, str(script)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    body = json.dumps({"input": "In one sentence: what does mortgage pre-qualification mean?"}).encode()
    for _ in range(40):
        time.sleep(1)
        try:
            req = urllib.request.Request("http://127.0.0.1:8088/responses", body, {"Content-Type": "application/json"})
            out = json.load(urllib.request.urlopen(req, timeout=60))
            break
        except OSError:
            continue
    text = "".join(c.get("text", "") for o in out.get("output", []) for c in o.get("content", []))
    print("HTTP /responses ->", redact(text))
finally:
    proc.terminate()

HTTP /responses -> Mortgage pre-qualification is an initial assessment of a borrower's financial situation to estimate how much they might be able to borrow.


## Key takeaways
- One system prompt (`MAPLE_INSTRUCTIONS`) gives consistent compliance behaviour across code, prompt and hosted agents.
- `AgentSession` = conversation state; stream for responsiveness.
- Prompt agents are versioned server-side assets, visible in the Foundry portal.
- Hosted agents = your code, run by Foundry; choose them when prompt agents are not expressive enough.

**Next up:** [02-tools-and-mcp.ipynb](02-tools-and-mcp.ipynb) - give the agent tools and MCP servers.